In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

# --- 2.0 Data Preparation ---

# Load the dataset
try:
    # Your CSV uses semicolons (;) as delimiters
    df = pd.read_csv('obesity.csv', delimiter=';')
    print("Dataset loaded successfully.")
    print("Original shape:", df.shape)
except FileNotFoundError:
    print("Error: 'obesity.csv' not found. Please ensure the file is in the correct directory.")
    exit()

# --- Data Cleaning & Restructuring ---

# 1. Clean up all column names to make them easier to work with
# This removes spaces and special characters
df.columns = df.columns.str.strip().str.replace(' = ', '_').str.replace(' ', '_')
print("\nCleaned column names:", df.columns.tolist())


# 2. Identify the one-hot encoded target columns
# These are all the columns that start with 'Obesity_'
obesity_cols = [col for col in df.columns if col.startswith('Obesity_')]
print(f"\nFound {len(obesity_cols)} one-hot encoded obesity columns.")


# 3. Create a single nominal target column from the one-hot encoded columns
# idxmax(axis=1) finds the column name with the '1' for each row
df['ObesityLevel_nominal'] = df[obesity_cols].idxmax(axis=1)
# Let's clean up the labels by removing the 'Obesity_' prefix
df['ObesityLevel_nominal'] = df['ObesityLevel_nominal'].str.replace('Obesity_', '')

print("\nCreated a single target column 'ObesityLevel_nominal':")
print(df['ObesityLevel_nominal'].head())


# --- 2.3 Data Transformation (Nominal to Numerical) ---
print("\n--- 2.3 Data Transformation ---")
# Now, we encode this new single column into numbers, as originally intended
label_encoder = LabelEncoder()
df['ObesityLevel_encoded'] = label_encoder.fit_transform(df['ObesityLevel_nominal'])

print("Transformed the single target column into numerical labels:")
print(df[['ObesityLevel_nominal', 'ObesityLevel_encoded']].head())
print("\nMapping of encoded values to original labels:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{i}: {class_name}")


# --- Prepare Final DataFrame for Splitting ---

# Features (X) are all columns EXCEPT the original one-hot encoded target columns
# and the new helper columns we created.
X = df.drop(columns=obesity_cols + ['ObesityLevel_nominal', 'ObesityLevel_encoded'])

# Target (y) is our new single, numerically-encoded column
y = df['ObesityLevel_encoded']


# --- 2.4 Data Export (Optional but good practice) ---
print("\n--- 2.4 Data Export ---")
# Let's create a final dataset for export
df_final_for_export = X.copy()
df_final_for_export['ObesityLevel'] = y # Add the single target column
df_final_for_export.to_csv('obesity_cleaned_and_restructured.csv', index=False)
print("Cleaned data has been exported to 'obesity_cleaned_and_restructured.csv'")


# --- 2.5 Sampling Strategy for Model Training and Testing ---
print("\n--- 2.5 Sampling Strategy ---")
# Split the final X and y into 70% training and 30% testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y  # Very important for classification!
)

print(f"Data split into training and testing sets.")
print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape:   {y.shape}")
print("-" * 20)
print(f"Training set shape (X_train): {X_train.shape}")
print(f"Testing set shape (X_test):  {X_test.shape}")

Dataset loaded successfully.
Original shape: (2111, 45)

Cleaned column names: ['Young', 'Adult', 'Senior', 'Gender_Female', 'Gender_Male', 'family_history_yes', 'family_history_no', 'FAVC_no', 'FAVC_yes', 'CAEC_Sometimes', 'CAEC_Frequently', 'CAEC_Always', 'CAEC_no', 'SMOKE_no', 'SMOKE_yes', 'SCC_no', 'SCC_yes', 'CALC_no', 'CALC_Sometimes', 'CALC_Frequently', 'CALC_Always', 'MTRANS_Public_Transportation', 'MTRANS_Walking', 'MTRANS_Automobile', 'MTRANS_Motorbike', 'MTRANS_Bike', 'Obesity_Normal_Weight', 'Obesity_Overweight_Level_I', 'Obesity_Overweight_Level_II', 'Obesity_Obesity_Type_I', 'Obesity_Insufficient_Weight', 'Obesity_Obesity_Type_II', 'Obesity_Obesity_Type_III', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'FCVC_up', 'NCP_up', 'CH2O_up', 'FAF_up', 'TUE_up']

Found 7 one-hot encoded obesity columns.

Created a single target column 'ObesityLevel_nominal':
0          Normal_Weight
1          Normal_Weight
2          Normal_Weight
3     Overweight_Level_I
4    Overwe

In [2]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from IPython.display import display, HTML

# --- 1. Data Preparation (Same as before) ---
df = pd.read_csv('obesity.csv', delimiter=';')
df.columns = df.columns.str.strip().str.replace(' = ', '_').str.replace(' ', '_')
obesity_cols = [col for col in df.columns if col.startswith('Obesity_')]
df['ObesityLevel_nominal'] = df[obesity_cols].idxmax(axis=1).str.replace('Obesity_', '')
label_encoder = LabelEncoder()
df['ObesityLevel_encoded'] = label_encoder.fit_transform(df['ObesityLevel_nominal'])
X = df.drop(columns=obesity_cols + ['ObesityLevel_nominal', 'ObesityLevel_encoded'])
y = df['ObesityLevel_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Scale Data and Train KNN Model 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)

# --- 3. Generate the Custom Report Table ---
class_names = label_encoder.classes_
cm = confusion_matrix(y_test, y_pred)
cm_transposed = cm.T
precision_scores = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_scores = recall_score(y_test, y_pred, average=None, zero_division=0)

df_report = pd.DataFrame(cm_transposed,
                         columns=[f"true {name}" for name in class_names],
                         index=[f"pred. {name}" for name in class_names])

df_report['class precision'] = [f"{score:.2%}" for score in precision_scores]
recall_row_data = [f"{score:.2%}" for score in recall_scores]
recall_row_data.append('')
df_report.loc['class recall'] = recall_row_data

# --- 4. Display the DataFrame as a Styled HTML Table ---
# CSS for styling the table to look professional
html_style = """
<style>
  .styled-table {
    border-collapse: collapse;
    margin: 25px 0;
    font-size: 0.9em;
    font-family: sans-serif;
    min-width: 400px;
    box-shadow: 0 0 20px rgba(0, 0, 0, 0.15);
    text-align: center;
  }
  .styled-table thead tr {
    background-color: #009879;
    color: #ffffff;
    text-align: center;
  }
  .styled-table th,
  .styled-table td {
    padding: 12px 15px;
    border: 1px solid #dddddd;
  }
  .styled-table tbody tr {
    border-bottom: 1px solid #dddddd;
  }
  .styled-table tbody tr:nth-of-type(even) {
    background-color: #f3f3f3;
  }
  .styled-table tbody tr:last-of-type {
    border-bottom: 2px solid #009879;
    font-weight: bold;
  }
  .styled-table th:first-child, .styled-table td:first-child {
    text-align: left;
    font-weight: bold;
  }
</style>
"""

# Convert the DataFrame to HTML and add the style
html_output = html_style + df_report.to_html(classes='styled-table', border=0)

print("\nResult of K-Nearest Neighbour Classification Model")
# Display the final HTML table
display(HTML(html_output))


Result of K-Nearest Neighbour Classification Model


,true Insufficient_Weight,true Normal_Weight,true Overweight_Level_I,true Overweight_Level_II,true Type_I,true Type_II,true Type_III,class precision
pred. Insufficient_Weight,75,21,6,3,1,2,0,69.44%
pred. Normal_Weight,5,36,8,4,3,0,1,63.16%
pred. Overweight_Level_I,2,13,54,9,8,1,0,62.07%
pred. Overweight_Level_II,0,9,11,52,7,1,1,64.20%
pred. Type_I,0,5,7,5,80,2,0,80.81%
pred. Type_II,0,2,1,14,6,83,0,78.30%
pred. Type_III,0,0,0,0,1,0,95,98.96%
class recall,91.46%,41.86%,62.07%,59.77%,75.47%,93.26%,97.94%,


In [3]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB # Import the correct model
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from IPython.display import display, HTML

# --- 1. Data Preparation (Same as before) ---
df = pd.read_csv('obesity.csv', delimiter=';')
df.columns = df.columns.str.strip().str.replace(' = ', '_').str.replace(' ', '_')
obesity_cols = [col for col in df.columns if col.startswith('Obesity_')]
df['ObesityLevel_nominal'] = df[obesity_cols].idxmax(axis=1).str.replace('Obesity_', '')
label_encoder = LabelEncoder()
df['ObesityLevel_encoded'] = label_encoder.fit_transform(df['ObesityLevel_nominal'])
X = df.drop(columns=obesity_cols + ['ObesityLevel_nominal', 'ObesityLevel_encoded'])
y = df['ObesityLevel_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)


# --- 2. Build and Train the Naive Bayes Model ---
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
print("Naive Bayes model trained.")


# 3. Make Predictions
y_pred = nb_model.predict(X_test)
print("Predictions made on the test set.")


# --- 4. Generate the Custom Report Table ---
class_names = label_encoder.classes_
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
cm_transposed = cm.T
precision_scores = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_scores = recall_score(y_test, y_pred, average=None, zero_division=0)

# Build the DataFrame for the report
df_report = pd.DataFrame(cm_transposed,
                         columns=[f"true {name}" for name in class_names],
                         index=[f"pred. {name}" for name in class_names])
df_report['class precision'] = [f"{score:.2%}" for score in precision_scores]
recall_row_data = [f"{score:.2%}" for score in recall_scores]
recall_row_data.append('')
df_report.loc['class recall'] = recall_row_data


# --- 5. Display the Styled HTML Table ---
html_style = """
<style>
  .styled-table { border-collapse: collapse; margin: 25px 0; font-size: 0.9em; font-family: sans-serif; min-width: 400px; box-shadow: 0 0 20px rgba(0, 0, 0, 0.15); text-align: center; }
  .styled-table thead tr { background-color: #009879; color: #ffffff; text-align: center; }
  .styled-table th, .styled-table td { padding: 12px 15px; border: 1px solid #dddddd; }
  .styled-table tbody tr { border-bottom: 1px solid #dddddd; }
  .styled-table tbody tr:nth-of-type(even) { background-color: #f3f3f3; }
  .styled-table tbody tr:last-of-type { border-bottom: 2px solid #009879; font-weight: bold; }
  .styled-table th:first-child, .styled-table td:first-child { text-align: left; font-weight: bold; }
</style>
"""
html_output = html_style + f"<h3>Overall Accuracy: {accuracy:.2%}</h3>" + df_report.to_html(classes='styled-table', border=0)

print("\nResult of Naïve Bayes Classification Model")
display(HTML(html_output))

Naive Bayes model trained.
Predictions made on the test set.

Result of Naïve Bayes Classification Model


,true Insufficient_Weight,true Normal_Weight,true Overweight_Level_I,true Overweight_Level_II,true Type_I,true Type_II,true Type_III,class precision
pred. Insufficient_Weight,78,53,21,16,0,0,0,46.43%
pred. Normal_Weight,0,17,7,1,1,0,0,65.38%
pred. Overweight_Level_I,3,7,13,1,2,0,0,50.00%
pred. Overweight_Level_II,0,3,2,16,3,2,1,59.26%
pred. Type_I,0,3,20,13,37,1,1,49.33%
pred. Type_II,1,3,24,40,62,86,0,39.81%
pred. Type_III,0,0,0,0,1,0,95,98.96%
class recall,95.12%,19.77%,14.94%,18.39%,34.91%,96.63%,97.94%,


In [4]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier # Import the correct model
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from IPython.display import display, HTML

# --- 1. Data Preparation (Same as before) ---
df = pd.read_csv('obesity.csv', delimiter=';')
df.columns = df.columns.str.strip().str.replace(' = ', '_').str.replace(' ', '_')
obesity_cols = [col for col in df.columns if col.startswith('Obesity_')]
df['ObesityLevel_nominal'] = df[obesity_cols].idxmax(axis=1).str.replace('Obesity_', '')
label_encoder = LabelEncoder()
df['ObesityLevel_encoded'] = label_encoder.fit_transform(df['ObesityLevel_nominal'])
X = df.drop(columns=obesity_cols + ['ObesityLevel_nominal', 'ObesityLevel_encoded'])
y = df['ObesityLevel_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)


# --- 2. Build and Train the Decision Tree Model ---
# NOTE: No data scaling is needed.
# We add random_state=42 for reproducibility of the results.
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model on the original (unscaled) training data
dt_model.fit(X_train, y_train)
print("Decision Tree model trained.")


# 3. Make Predictions
y_pred = dt_model.predict(X_test)
print("Predictions made on the test set.")


# --- 4. Generate the Custom Report Table ---
class_names = label_encoder.classes_
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
cm_transposed = cm.T
precision_scores = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_scores = recall_score(y_test, y_pred, average=None, zero_division=0)

# Build the DataFrame for the report
df_report = pd.DataFrame(cm_transposed,
                         columns=[f"true {name}" for name in class_names],
                         index=[f"pred. {name}" for name in class_names])
df_report['class precision'] = [f"{score:.2%}" for score in precision_scores]
recall_row_data = [f"{score:.2%}" for score in recall_scores]
recall_row_data.append('')
df_report.loc['class recall'] = recall_row_data


# --- 5. Display the Styled HTML Table ---
html_style = """
<style>
  .styled-table { border-collapse: collapse; margin: 25px 0; font-size: 0.9em; font-family: sans-serif; min-width: 400px; box-shadow: 0 0 20px rgba(0, 0, 0, 0.15); text-align: center; }
  .styled-table thead tr { background-color: #009879; color: #ffffff; text-align: center; }
  .styled-table th, .styled-table td { padding: 12px 15px; border: 1px solid #dddddd; }
  .styled-table tbody tr { border-bottom: 1px solid #dddddd; }
  .styled-table tbody tr:nth-of-type(even) { background-color: #f3f3f3; }
  .styled-table tbody tr:last-of-type { border-bottom: 2px solid #009879; font-weight: bold; }
  .styled-table th:first-child, .styled-table td:first-child { text-align: left; font-weight: bold; }
</style>
"""
html_output = html_style + f"<h3>Overall Accuracy: {accuracy:.2%}</h3>" + df_report.to_html(classes='styled-table', border=0)

print("\nResult of Decision Tree Classification Model")
display(HTML(html_output))

Decision Tree model trained.
Predictions made on the test set.

Result of Decision Tree Classification Model


,true Insufficient_Weight,true Normal_Weight,true Overweight_Level_I,true Overweight_Level_II,true Type_I,true Type_II,true Type_III,class precision
pred. Insufficient_Weight,74,4,0,0,0,0,0,94.87%
pred. Normal_Weight,8,73,5,0,0,0,0,84.88%
pred. Overweight_Level_I,0,8,74,1,0,0,0,89.16%
pred. Overweight_Level_II,0,1,8,80,2,0,0,87.91%
pred. Type_I,0,0,0,5,102,1,0,94.44%
pred. Type_II,0,0,0,1,2,87,1,95.60%
pred. Type_III,0,0,0,0,0,1,96,98.97%
class recall,90.24%,84.88%,85.06%,91.95%,96.23%,97.75%,98.97%,


In [6]:
#1.3.2

# Prepare target variable (Age Group)
df_age_task['AgeGroup_nominal'] = df_age_task[['Young', 'Adult', 'Senior']].idxmax(axis=1)
y = LabelEncoder().fit_transform(df_age_task['AgeGroup_nominal'])

# Drop age & obesity columns to avoid bias
X = df_age_task.drop(columns=['Young', 'Adult', 'Senior'] + obesity_cols + ['AgeGroup_nominal'])

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_train_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit(X_train).transform(X_test)

# Train and evaluate KNN model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)


bayes

# Create target: Age group from one-hot encoded columns
df_age_task['AgeGroup_nominal'] = df_age_task[['Young', 'Adult', 'Senior']].idxmax(axis=1)
y = LabelEncoder().fit_transform(df_age_task['AgeGroup_nominal'])

# Drop irrelevant columns (age and obesity info)
X = df_age_task.drop(columns=['Young', 'Adult', 'Senior'] + obesity_cols + ['AgeGroup_nominal'])

# Train/test split (70:30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Train and predict using Gaussian Naïve Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)



# Create target from one-hot encoded age columns
df_age_task['AgeGroup_nominal'] = df_age_task[['Young', 'Adult', 'Senior']].idxmax(axis=1)
y = LabelEncoder().fit_transform(df_age_task['AgeGroup_nominal'])

# Drop irrelevant columns (age + obesity info)
X = df_age_task.drop(columns=['Young', 'Adult', 'Senior'] + obesity_cols + ['AgeGroup_nominal'])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Train and predict using Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)


In [12]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from IPython.display import display, HTML

# --- 1. Load and Prepare Data for Age Group Analysis ---
# Load the pre-processed CSV file
# Use a new DataFrame name to avoid conflicts with other cells
df_age_task = pd.read_csv('obesity.csv', delimiter=';')

# Clean column names (good practice)
df_age_task.columns = df_age_task.columns.str.strip().str.replace(' ', '_')

# --- 2. Create Target (y) and Features (X) from the One-Hot Encoded Data ---

# Identify the one-hot encoded AGE columns
age_cols = ['Young', 'Adult', 'Senior'] 

# Create a single nominal target column from these columns
df_age_task['AgeGroup_nominal'] = df_age_task[age_cols].idxmax(axis=1)

# Define the target 'y' and encode it numerically
age_encoder = LabelEncoder()
y = age_encoder.fit_transform(df_age_task['AgeGroup_nominal'])

# Define features 'X' by dropping ALL age-related and obesity-related columns
# We drop obesity columns to prevent the model from simply learning that Obesity_Type_III is linked to 'Young' etc.
obesity_cols = [col for col in df_age_task.columns if col.startswith('Obesity_')]
cols_to_drop = age_cols + obesity_cols + ['AgeGroup_nominal']
X = df_age_task.drop(columns=cols_to_drop)

# --- 3. Split Data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# --- 4. Scale Data and Train KNN Model ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)

# --- 5. Generate and Display the Report Table ---
class_names = age_encoder.classes_
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=range(len(class_names)))
cm_transposed = cm.T
precision_scores = precision_score(y_test, y_pred, average=None, zero_division=0, labels=range(len(class_names)))
recall_scores = recall_score(y_test, y_pred, average=None, zero_division=0, labels=range(len(class_names)))

df_report = pd.DataFrame(cm_transposed,
                         columns=[f"true {name}" for name in class_names],
                         index=[f"pred. {name}" for name in class_names])

df_report['class precision'] = [f"{score:.2%}" for score in precision_scores]
recall_row_data = [f"{score:.2%}" for score in recall_scores]
recall_row_data.append('')
df_report.loc['class recall'] = recall_row_data

html_style = """
<style>
  .styled-table { border-collapse: collapse; margin: 25px 0; font-size: 0.9em; font-family: sans-serif; min-width: 400px; box-shadow: 0 0 20px rgba(0, 0, 0, 0.15); text-align: center; }
  .styled-table thead tr { background-color: #009879; color: #ffffff; text-align: center; }
  .styled-table th, .styled-table td { padding: 12px 15px; border: 1px solid #dddddd; }
  .styled-table tbody tr { border-bottom: 1px solid #dddddd; }
  .styled-table tbody tr:nth-of-type(even) { background-color: #f3f3f3; }
  .styled-table tbody tr:last-of-type { border-bottom: 2px solid #009879; font-weight: bold; }
  .styled-table th:first-child, .styled-table td:first-child { text-align: left; font-weight: bold; }
</style>
"""
html_output = html_style + f"<h3>Overall Accuracy: {accuracy:.2%}</h3>" + df_report.to_html(classes='styled-table', border=0)

print("\nResult of K-Nearest Neighbour Classification Model for Age Group")
display(HTML(html_output))


Result of K-Nearest Neighbour Classification Model for Age Group


,true Adult,true Senior,true Young,class precision
pred. Adult,88,3,17,81.48%
pred. Senior,1,1,0,50.00%
pred. Young,23,0,501,95.61%
class recall,78.57%,25.00%,96.72%,


In [14]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from IPython.display import display, HTML

# --- 1. Load and Prepare Data Freshly ---
df_age_task = pd.read_csv('obesity.csv', delimiter=';')
df_age_task.columns = df_age_task.columns.str.strip().str.replace(' ', '_')

# --- 2. Create Target (y) and Features (X) from the One-Hot Encoded Data ---
age_cols = ['Young', 'Adult', 'Senior'] 
df_age_task['AgeGroup_nominal'] = df_age_task[age_cols].idxmax(axis=1)

age_encoder = LabelEncoder()
y = age_encoder.fit_transform(df_age_task['AgeGroup_nominal'])

obesity_cols = [col for col in df_age_task.columns if col.startswith('Obesity_')]
cols_to_drop = age_cols + obesity_cols + ['AgeGroup_nominal']
X = df_age_task.drop(columns=cols_to_drop)

# --- 3. Split Data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# --- 4. Build and Train the Naive Bayes Model ---
# No scaling is needed
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
y_pred = nb_model.predict(X_test)

# --- 5. Generate and Display the Report Table ---
class_names = age_encoder.classes_
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=range(len(class_names)))
cm_transposed = cm.T
precision_scores = precision_score(y_test, y_pred, average=None, zero_division=0, labels=range(len(class_names)))
recall_scores = recall_score(y_test, y_pred, average=None, zero_division=0, labels=range(len(class_names)))

df_report = pd.DataFrame(cm_transposed,
                         columns=[f"true {name}" for name in class_names],
                         index=[f"pred. {name}" for name in class_names])

df_report['class precision'] = [f"{score:.2%}" for score in precision_scores]
recall_row_data = [f"{score:.2%}" for score in recall_scores]
recall_row_data.append('')
df_report.loc['class recall'] = recall_row_data

html_style = """
<style>
  .styled-table { border-collapse: collapse; margin: 25px 0; font-size: 0.9em; font-family: sans-serif; min-width: 400px; box-shadow: 0 0 20px rgba(0, 0, 0, 0.15); text-align: center; }
  .styled-table thead tr { background-color: #009879; color: #ffffff; text-align: center; }
  .styled-table th, .styled-table td { padding: 12px 15px; border: 1px solid #dddddd; }
  .styled-table tbody tr { border-bottom: 1px solid #dddddd; }
  .styled-table tbody tr:nth-of-type(even) { background-color: #f3f3f3; }
  .styled-table tbody tr:last-of-type { border-bottom: 2px solid #009879; font-weight: bold; }
  .styled-table th:first-child, .styled-table td:first-child { text-align: left; font-weight: bold; }
</style>
"""
html_output = html_style + f"<h3>Overall Accuracy: {accuracy:.2%}</h3>" + df_report.to_html(classes='styled-table', border=0)

print("\nResult of Naïve Bayes Classification Model for Age Group")
display(HTML(html_output))


Result of Naïve Bayes Classification Model for Age Group


,true Adult,true Senior,true Young,class precision
pred. Adult,90,2,319,21.90%
pred. Senior,16,2,84,1.96%
pred. Young,6,0,115,95.04%
class recall,80.36%,50.00%,22.20%,


In [16]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from IPython.display import display, HTML

# --- 1. Load and Prepare Data Freshly ---
df_age_task = pd.read_csv('obesity.csv', delimiter=';')
df_age_task.columns = df_age_task.columns.str.strip().str.replace(' ', '_')

# --- 2. Create Target (y) and Features (X) from the One-Hot Encoded Data ---
age_cols = ['Young', 'Adult', 'Senior'] 
df_age_task['AgeGroup_nominal'] = df_age_task[age_cols].idxmax(axis=1)

age_encoder = LabelEncoder()
y = age_encoder.fit_transform(df_age_task['AgeGroup_nominal'])

obesity_cols = [col for col in df_age_task.columns if col.startswith('Obesity_')]
cols_to_drop = age_cols + obesity_cols + ['AgeGroup_nominal']
X = df_age_task.drop(columns=cols_to_drop)

# --- 3. Split Data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# --- 4. Build and Train the Decision Tree Model ---
# No scaling is needed
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)

# --- 5. Generate and Display the Report Table ---
class_names = age_encoder.classes_
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=range(len(class_names)))
cm_transposed = cm.T
precision_scores = precision_score(y_test, y_pred, average=None, zero_division=0, labels=range(len(class_names)))
recall_scores = recall_score(y_test, y_pred, average=None, zero_division=0, labels=range(len(class_names)))

df_report = pd.DataFrame(cm_transposed,
                         columns=[f"true {name}" for name in class_names],
                         index=[f"pred. {name}" for name in class_names])

df_report['class precision'] = [f"{score:.2%}" for score in precision_scores]
recall_row_data = [f"{score:.2%}" for score in recall_scores]
recall_row_data.append('')
df_report.loc['class recall'] = recall_row_data

html_output = html_style + f"<h3>Overall Accuracy: {accuracy:.2%}</h3>" + df_report.to_html(classes='styled-table', border=0)

print("\nResult of Decision Tree Classification Model for Age Group")
display(HTML(html_output))


Result of Decision Tree Classification Model for Age Group


,true Adult,true Senior,true Young,class precision
pred. Adult,95,3,18,81.90%
pred. Senior,0,0,2,0.00%
pred. Young,17,1,498,96.51%
class recall,84.82%,0.00%,96.14%,
